# 02 · Construcción del dataset de evaluación Speech-to-Speech

**Entrada:** `data/processed/manifests/fleurs_sample.csv`

**Salida:** `data/processed/manifests/evaluation_v1.csv`

Convierte la muestra de FLEURS en el primer **golden dataset** del proyecto: audio + transcripción en español + traducción de referencia en inglés + split `dev/test`.

**Regla importante:** la traducción de referencia (ground truth) se introduce **manualmente**. Nunca debe generarse con el modelo de traducción que después será evaluado.

## 1. Entorno y dependencias

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT_OVERRIDE: Path | None = None

def _is_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

def _is_project_root(path: Path) -> bool:
    return (path / 'src').is_dir() and (path / 'requirements' / 'dataset.txt').is_file()

def _find_drive_project_root() -> Path:
    if PROJECT_ROOT_OVERRIDE is not None:
        return PROJECT_ROOT_OVERRIDE.expanduser().resolve()
    matches = sorted({src_dir.parent for src_dir in Path('/content/drive').glob('**/src')
                      if _is_project_root(src_dir.parent)})
    if len(matches) == 1:
        return matches[0]
    if matches:
        found = '\n - '.join(str(path) for path in matches)
        raise SystemExit(f'Se encontraron varios proyectos:\n - {found}\nAsigna uno a PROJECT_ROOT_OVERRIDE.')
    raise SystemExit('No se encontro IA-Proyecto. Monta la cuenta correcta de Drive o asigna PROJECT_ROOT_OVERRIDE.')

if _is_colab():
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = _find_drive_project_root()
else:
    PROJECT_ROOT = Path.cwd()

if not _is_project_root(PROJECT_ROOT):
    raise SystemExit(f'No se encontro la raiz del proyecto en {PROJECT_ROOT}.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)


In [ ]:
%pip install --quiet -r requirements/dataset.txt

### 1.1 Imports y rutas

In [ ]:
import pandas as pd
from IPython.display import display

from src.dataset import manifest as m

SEED = m.SEED
DEV_RATIO = m.DEV_RATIO
MANIFEST_PATH = m.FLEURS_MANIFEST_PATH
OUTPUT_PATH = m.EVALUATION_CSV
MANUAL_PATH = m.MANUAL_TRANSLATIONS_PATH

## 2. Cargar el manifest de FLEURS

In [ ]:
manifest = m.load_manifest(MANIFEST_PATH)
print(f"Registros cargados: {len(manifest)}")
display(manifest)

## 3. Traducciones de referencia manuales

Flujo recomendado:
1. La siguiente celda crea (si no existe) `data/processed/manifests/manual_translations.csv` con una fila por muestra.
2. En Colab se descarga ese CSV, se completa la columna `translation_en` y se vuelve a subir a la misma ruta.
3. Alternativamente, se pueden escribir las traducciones directamente en el dict de la sección 3.2.

El CSV **no se sobrescribe** en ejecuciones posteriores: solo se crea la primera vez.

In [ ]:
manual_path = m.ensure_manual_translations_file(manifest)
print("Archivo de traducciones:", manual_path)

In [ ]:
try:
    from google.colab import files
    files.download(str(manual_path))
    print("Descarga el CSV, complétalo y súbelo de vuelta a", manual_path)
except ImportError:
    print("No estás en Colab: edita el CSV directamente con tu editor.")

### 3.1 Cargar traducciones desde el CSV manual

In [ ]:
translations = m.load_translations_dict()
print(f"Traducciones cargadas del CSV: {len(translations)} de {len(manifest)}")

### 3.2 (Opcional) Traducciones directas en el notebook

Útil para pocas muestras. Descomenta una línea si quieres traducir aquí mismo.

In [ ]:
manual_translations_extra: dict[str, str] = {}
# manual_translations_extra = {"fleurs_es_419_0000": "The dog is sleeping."}

### 3.3 Aplicar traducciones al manifest

In [ ]:
translations.update(manual_translations_extra)
df = m.apply_translations(manifest, translations)
missing = df[df["translation_en"].str.strip() == ""]
print(f"Traducciones aplicadas: {len(translations)} de {len(df)}")
display(missing[["sample_id", "transcription_es", "translation_en"]])

Si quedan filas sin traducir, la siguiente celda detendrá la ejecución: completa el CSV y vuelve a ejecutar el notebook desde la sección 3.

In [ ]:
m.assert_complete_evaluation(df, audio_root=PROJECT_ROOT)
print("Todos los campos obligatorios están completos: audio, transcripción, traducción y duración.")

## 4. Asignar split dev/test (70% / 30%)

La división es reproducible (semilla fija). Si existiera `speaker_id` se separarían grupos completos de hablante para evitar *leakage*; FLEURS no lo expone, así que aquí la división es por filas y así queda documentado.

In [ ]:
df = m.assign_splits(df, dev_ratio=DEV_RATIO, seed=SEED)
display(m.split_distribution(df))

## 5. Estadísticas del dataset final

In [ ]:
display(m.evaluate_manifest_stats(df))

## 6. Guardar el dataset de evaluación

Se ordenan las columnas canónicas del proyecto y se guarda en CSV (UTF-8).

In [ ]:
m.save_evaluation_manifest(df, OUTPUT_PATH)
display(df)

**Resultado esperado de esta fase:** `evaluation_v1.csv` es el dataset dorado con 20 muestras, listo para ser validado en el notebook 03. Su objetivo posterior es evaluar ASR (WER), traducción (BLEU/COMET) y voice cloning; **no** es el corpus de entrenamiento del traductor.